# Tutorial on time of arrival estimation under jitter noise

## Introduction

Time of Arrival (TOA) estimation is a fundamental problem in localization. It involves determining the precise moment a signal, or a characteristic event within a signal, arrives at a sensor or detector. However, signals are often corrupted by various forms of uncertainty, collectively known as **jitter**. Jitter can be random or deterministic deviations in the timing of events from their ideal or expected instances. These timing variations can significantly degrade the accuracy of TOA estimates and, consequently, the performance of systems that rely on them.

This tutorial explores the concepts of correlation, its application in TOA estimation, and how **random** jitter influences this process. We will delve into the mathematical foundations and analyze the effects of applying cross-correlation for TOA estimation under random jitter. The case study will apply this function to micro electrode array processing on brain signals.

### Types of Jitter: Deterministic and Random

Timing jitter for signal processing can be classified into 2 main categories based on its statistical properties and underlying causes. This document primarily focuses on random jitter.



#### 1. Deterministic Jitter (DJ)

Deterministic jitter can be caused by crosstalk interference, 50/60 Hz power supply noise, and multipath. It is characterized as a signal that is bounded, predictable, and may be either periodic or aperiodic. Periodic jitter is easily visible in frequency domain.

---
#### 2. Random Jitter (RJ)

Random jitter is primarily caused by thermal noise. It is characterized as a signal contaminated by an unbounded probabilistic distribution - typically a gaussian distribution.

### Other causes of jitter
Below are other potential causes of jitter, under the lens of timing applications for surgically placed brain micro electrode arrays (MEA). However, these will not be expanded upon for this tutorial. We believe that interference from other signals would be the main core possible effects on jitter to investigate for brain signals.


#### 1. Interference / Crosstalk

Different propagating signals (other brain activity, artifacts) can affect array electrodes differently. Unlike thermal noise, interference is structured and may have spatial coherence.

Potential mitigation techniques include

**MVDR (Minimum Variance Distortionless Response):**
- Adaptive beamforming technique that attempts to minimize output variance while maintaining unity gain in the direction of interest
- Requires knowledge or estimation of the signal direction and interference/noise covariance

**MUSIC (Multiple Signal Classification):**
- Uses eigendecomposition to separate signal and noise subspaces
- Can estimate direction of arrival for multiple signals
- Requires multiple snapshots and assumptions about the number of signal sources

#### 2. Multipath Propagation

Multipath refers to the phenomenon where the transmitted signal reaches the receiver via multiple paths due to reflections, creating multiple delayed and attenuated copies of the signal.

#### 3. Synchronization Errors

Sampling oscillators between electrode channels must be synchronized to align receiver timing with transmitter. Imperfect estimation leads to residual timing errors. However, for multiple electrode systems, we can assume electrode channels are built to share one oscillator.

#### 4. Oscillator Instability

Clocks in signal processing systems can drift over time. However, as we should assume the electrode system shares one oscillator, they all receive the same drift. For many position finding frameworks, the exact arrival time is not required and only the relative arrival time must be accurate.

#### 5. ADC Aperture Jitter

Analog-to-Digital Converters can sample at slightly incorrect times due to aperture jitter however, ADC jitter is primarily significant for high-frequency signals or high sampling rate systems. Electrode collection systems rarely go beyond 2 KHz, as much of the signals carry information at lower frequencies.

---

#### Side Notes

As electrodes degrade, their impedance tends to increase, potentially creating a frequency-dependent filtering effect. This degradation may disproportionately attenuate high-frequency signal components compared to low-frequency ones. It's worth investigating whether electrode aging can affect detection performance.



## Random Jitter and Correlation

### Maximum Likelihood Time Delay Estimation

Given that our noise is thermal noise, we can construct the maximum likelihood estimator for estimating time delay. Below, we define the received signal as:

**Received signal:**
$$
r[n] = s[n - \tau] + w[n]
$$

**Noise:**
$$
w[n] \sim \mathcal{N}(0, \sigma^2)
$$

where $w[n]$ is white Gaussian noise.

For i.i.d gaussian noise, it is well known that the ML estimator minimizes the squared error between the received signal and the original:

$$
\hat{\tau}_{ML} = \arg\min_{\tau} \sum_{n} \left[ r[n] - s[n - \tau] \right]^2
$$

For those who read my LS-RLS tutorial, you might notice that this has a similar for as batch least squares. However, it's important to note that the parameter to estimate $\tau$ is **nonlinearly** related in the equation to $s[n]$. Therefore, we cannot simply apply the linear LS solution of the psuedo-inverse.

However, rather than solve this iteratively using nonlinear LS solution methods, we can expanding the squared term:

$$
\sum_{n} \left[ r[n] - s[n - \tau] \right]^2 = \sum_{n} r^2[n] - 2\sum_{n} r[n] \, s[n - \tau] + \sum_{n} s^2[n - \tau]
$$

Given this expansion, we note the follow observations

- $\sum_{n} r^2[n]$: is independent of $\tau$
- $\sum_{n} s^2[n - \tau] = E_s$: is the signal energy, which is a constant
- $-2\sum_{n} r[n] \, s[n - \tau]$: depends on $\tau$

Therefore, minimizing the squared error is equivalent to **maximizing the cross-correlation**:

$$
\boxed{\hat{\tau}_{ML} = \arg\max_{\tau} \sum_{n} r[n] \, s[n - \tau]}
$$

$$
\boxed{\hat{\tau}_{ML} = \arg\max_{\tau} R_{rs}(\tau)}
$$

This is the **matched filter** operation: $s[n]$ acts as the filter matched to the received noisy signal $r[n]$, and the delay $\tau$ that maximizes the output is the ML estimate.

### Threshold Peak Detection

https://ieeexplore.ieee.org/document/1545871$0

Given that we know how to estimate the time delay, we also need a method of detecting if a signal is in AWGN. This can be construed as a neyman-pearson detection problem.

We define two hypothesis tests, where:



## What Is Correlation?

In the prior section, we discussed how correlation between the original signal s[n] and the contaminated signal is the ML estimate for estimating the time delay. Time-of-arrival (ToA) problems require knowing when a signal arrived given by estimating the timing offset using correlation. Peak detection is used to determine relative delay between electrodes in an array.

Correlation measures the similarity between two signals as a function of time lag. It quantifies how much one signal resembles another (or itself) when shifted by a delay $\tau$.

For continuous-time signals $r(t)$ and $s(t)$, the cross-correlation function is:

$$
R_{rs}(\tau) = \int_{-\infty}^{\infty} r(t) s^*(t - \tau) \, dt
$$

For discrete-time signals (sampled data):

$$
R_{rs}[m] = \sum_{n=-\infty}^{\infty} r[n] s^*[n - m]
$$

where $s^*$ denotes the complex conjugate (for real signals, $s^* = s$) Moving fowards, we assume this signal is **real** and **discrete**.

### Cross-Correlation vs. Auto-Correlation
## TODO IS THIS EVEN USEFUL? HOW DOES THIS RELATED TO DET THEORY?

#### Auto-Correlation: The Ground Truth Reference

Auto-correlation correlates a signal with a delayed version of itself:

$$
R_{ss}[\tau] = \sum_{n=-\infty}^{\infty} s[n] s[n - \tau]
$$

In the **absence of noise**, the auto-correlation of s[n] represents the ideal correlation function.
It shows the inherent timing resolution of the signal, the peak width is a function of the signal's bandwidth, and it serves as a **ground truth** for the shape of perfect correlation.

This can be important for analysis as certain signals may contain multiple but smaller peaks, which under noise, can become false positive detections.

---

### Cross-Correlation: Scoring Against the Ground Truth

To create a score, we can apply the normalized cross-correlation:

$$
\rho_{rs}(\tau) = \frac{R_{rs}(\tau)}{\sqrt{R_{rr}(0) \cdot R_{ss}(0)}}
$$

Or if s[n] and r[n] are zero mean:

$$
\rho_{rs}(\tau) = \frac{R_{rs}(\tau)}{\sigma_{r} \sigma_{s} }
$$

This ensures $-1 \leq \rho_{rs}(\tau) \leq 1$, where:
- $\rho = 1$ indicates perfect match
- $\rho = 0$ indicates no correlation
- $\sigma_{r}$, $\sigma_{r}$ are the respective standard deviations for each signal.


## Signal characteristics that affect correlation under AWGN



### Correlation Peak Width and Bandwidth

## TODO: THIS HAS TO BE RELATED TO CRLB. CALC CRLB using FISHER INFO.

The width of the correlation peak determines timing resolution:

**Relationship to Bandwidth:**

The main lobe width of $R_{ss}(\tau)$ is inversely proportional to signal bandwidth $B$:

$$
\Delta \tau_{\text{width}} \approx \frac{1}{B}
$$

**Implications:**

- **Wideband signals** (large $B$) → narrow correlation peak → precise timing
- **Narrowband signals** (small $B$) → broad correlation peak → poor timing resolution
- This is why GPS and radar use wideband or spread-spectrum signals

**Example:**
- 1 kHz bandwidth signal: $\Delta \tau \approx 1$ ms resolution
- 10 MHz bandwidth signal: $\Delta \tau \approx 0.1$ μs resolution

---

#### Noise Effects on Peak Detection

**In Presence of Noise:**

If $r(t) = s(t - t_0) + n(t)$ where $n(t)$ is noise:

$$
R_{rs}(\tau) = R_{ss}(\tau - t_0) + R_{ns}(\tau)
$$

The noise term $R_{ns}(\tau)$ causes the peak location to fluctuate, introducing timing jitter.

**Timing Variance:**

The Cramér-Rao lower bound for timing estimation variance is:

$$
\sigma_{\tau}^2 \geq \frac{1}{8\pi^2 \beta^2 \cdot \text{SNR}}
$$

where $\beta^2$ is the mean-square bandwidth of the signal:

$$
\beta^2 = \frac{\int_{-\infty}^{\infty} f^2 |S(f)|^2 \, df}{\int_{-\infty}^{\infty} |S(f)|^2 \, df}
$$

**Key Insights:**
- Timing accuracy improves with higher SNR
- Timing accuracy improves with larger bandwidth (higher $\beta$)
- There is a fundamental limit imposed by noise

## Applied Considerations for Noise-Induced Jitter


### Threshold Detection

To distinguish true peaks from noise fluctuations:

**Setting Thresholds:**
- Set threshold based on expected SNR and noise statistics
- Use statistical detection theory (e.g., Neyman-Pearson criterion)
- False alarm rate depends on threshold choice

**Trade-offs:**
- **High threshold:** Fewer false alarms, but may miss weak signals (missed detections)
- **Low threshold:** Better detection of weak signals, but more false alarms
- Optimal threshold balances these competing concerns based on application requirements

**1. Averaging Multiple Measurements:**

When multiple independent observations are available:

$$
\sigma_{\tau,\text{avg}}^2 = \frac{\sigma_{\tau}^2}{N}
$$

Averaging $N$ measurements reduces timing jitter by factor $\sqrt{N}$.

**2. SNR Estimation:**

Knowing the noise level helps set appropriate detection thresholds:
- Estimate noise from signal-free regions
- Use known signal characteristics to predict expected peak height
- Adaptive thresholds adjust based on local noise conditions

**3. Peak Quality Metrics:**

Beyond simple peak detection, assess peak quality:
- **Peak-to-sidelobe ratio:** Higher ratios indicate cleaner detection
- **Peak sharpness:** Narrower peaks suggest less noise corruption
- **Confidence intervals:** Quantify uncertainty in timing estimate based on peak shape

### Jitter Effects on Correlation Peaks

### Observing Correlation Degradation

### Estimating Jitter with Correlation Techniques

# Case Study: Correlation between channels to detect neurological signals and signatures that exist between channels